# All Fastchem Gas and Cond

Hajime Kawahara 2025/11/27



In [1]:
from jax import config
config.update("jax_enable_x64", True)

We assume N2+H2O (gas, water, ice) system using fastchem/fastchem_cond presets. 

In [8]:
from exogibbs.presets.fastchem_cond import chemsetup as condsetup
cond = condsetup()
from exogibbs.presets.fastchem import chemsetup as gassetup
gas = gassetup()


fastchem_cond presets in ExoGibbs
number of species: 186 elements: 28 molecules: 186
fastchem presets in ExoGibbs
number of species: 523 elements: 28 molecules: 495


In [15]:
import numpy as np
from exojax.utils.zsol import nsol
import jax.numpy as jnp

solar_abundance = nsol()
nsol_vector = jnp.array(
    [solar_abundance[el] for el in gas.elements[:-1]]
)  # no solar abundance for e-
element_vector = jnp.append(nsol_vector, 0.0)

formula_matrix_gas = gas.formula_matrix

print("Formula matrix (gas):")
print(formula_matrix_gas)

formula_matrix_cond = cond.formula_matrix

print("Formula matrix (cond):")
print(formula_matrix_cond)

b_ref = gas.element_vector_reference

Database for solar abundance =  AAG21
Asplund, M., Amarsi, A. M., & Grevesse, N. 2021, arXiv:2105.01661
Formula matrix (gas):
[[ 1  0  0 ...  0  0  0]
 [ 0  1  0 ...  0  0  0]
 [ 0  0  1 ...  0  0  0]
 ...
 [ 0  0  0 ...  1  0  0]
 [ 0  0  0 ...  0  1  1]
 [ 0  0  0 ...  1 -1  1]]
Formula matrix (cond):
[[1 1 1 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 1 1 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


This setting yields rank(Ac, Ag) < |b_element| because formula_matrix_gas[:,0] = formula_matrix_cond[:,0]. We need to redefine the formulation using the matrix contraction. 

In [13]:
from exogibbs.thermo.stoichiometry import contract_formula_matrix
formula_matrix_gas_eff, formula_matrix_cond_eff, indep_element_mask = contract_formula_matrix(formula_matrix_gas, formula_matrix_cond)
#elements_eff =elements[indep_element_mask]

print("Formula matrix (gas):")
print(formula_matrix_gas_eff)
print("Formula matrix (cond):")
print(formula_matrix_cond_eff)
#print("independent elements:")
#print(elements_eff)


No contraction of the system needed.
Formula matrix (gas):
[[ 1  0  0 ...  0  0  0]
 [ 0  1  0 ...  0  0  0]
 [ 0  0  1 ...  0  0  0]
 ...
 [ 0  0  0 ...  1  0  0]
 [ 0  0  0 ...  0  1  1]
 [ 0  0  0 ...  1 -1  1]]
Formula matrix (cond):
[[1 1 1 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 1 1 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


Output the reference-state value of ( $h = \mu / (RT)$ ) at temperature ( T ).


## minimization using minimize_gibbs_cond_core

In [14]:
from exogibbs.optimize.minimize_cond import minimize_gibbs_cond_core
import jax.numpy as jnp
from exogibbs.api.chemistry import ThermoState

from exogibbs.optimize.core import compute_ln_normalized_pressure


In [26]:

# Thermodynamic conditions
P = 1.0  # bar
Pref = 1.0  # bar, reference pressure
ln_normalized_pressure = compute_ln_normalized_pressure(P, Pref)

# Initial guess for log number densities
ln_etak = jnp.zeros(formula_matrix_cond_eff.shape[1])  # log(eta)
ln_nk = jnp.zeros(formula_matrix_gas_eff.shape[1])  # log(n_species)
ln_mk = jnp.zeros(formula_matrix_cond_eff.shape[1])  # log(n_condensates)
ln_ntot = 0.0  # log(total number density)

epsilon = -20.0
for i, temperature in enumerate([200.0]):

    thermo_state = ThermoState(
        temperature=temperature,
        ln_normalized_pressure=ln_normalized_pressure,
        element_vector=b_ref,
    )

    ln_nk, ln_mk, ln_etak, ln_ntot, counter = minimize_gibbs_cond_core(
        thermo_state,
        ln_nk_init=ln_nk,
        ln_mk_init=ln_mk,
        ln_etak_init=ln_etak,
        ln_ntot_init=ln_ntot,
        formula_matrix=formula_matrix_gas_eff,
        formula_matrix_cond=formula_matrix_cond_eff,
        hvector_func=gas.hvector_func,
        hvector_cond_func=cond.hvector_func,
        epsilon=epsilon,  ### new argument
        residual_crit=1.0e-11,
        max_iter=1,
    )


    print(jnp.exp(ln_etak), jnp.exp(ln_nk), jnp.exp(ln_mk), jnp.exp(ln_ntot), jnp.exp(epsilon))
    

[2.70766900e+09 6.57521733e-01 1.91312030e+05 1.21141988e+03
 1.59322435e+05 8.63129171e+00 1.00858848e+06 8.70454693e-05
 1.15719918e-08 3.33637121e-06 1.23840152e+00 1.44822585e-01
 5.17284861e+04 2.01739530e-08 1.69231838e-06 2.45063034e-10
 3.24929113e+04 3.32036767e-16 9.44269547e-25 5.48832940e-02
 2.62778592e+00 1.71746724e+00 2.09988091e-01 4.28486279e-04
 3.20901012e+01 1.68216976e+01 2.47380114e+04 1.29421712e+03
 3.21631172e-03 5.95725121e+07 6.30696013e+28 1.66038917e-03
 2.09892019e+14 5.60535866e+03 1.60294491e+05 1.03576023e+01
 1.72192451e+11 8.37648293e-01 3.22983183e-04 2.58362989e+01
 1.04040542e+01 5.92365447e-01 3.39059205e+01 9.43079852e+01
 2.45131358e+27 1.60713698e-01 2.50149180e+24 3.82880683e+00
 1.98779054e+26 2.24060534e+01 3.55856220e+04 3.54086136e+01
 5.43509002e+00 4.39042173e+02 2.86452701e+08 9.88273421e+08
 1.30233730e+05 6.60863346e+04 2.22060986e+04 4.43302994e+06
 1.34868061e+00 5.60713924e-01 1.93987670e+05 1.38289553e+00
 6.86641893e+02 9.552415